## **Chapter 7 - Activating AI Agents**
This notebook explores Agentic AI through structured examples, showcasing how AI agents plan, retrieve data, and execute tasks autonomously. Using `LangChain` and `CrewAI`, the notebook demonstrates key abstractions, including datasets, prompts, model selection, agent-based reasoning, and task execution. Examples range from structured data retrieval using Hugging Face datasets to interactive AI agents that process information and act independently. By the end, you'll see how these frameworks power AI-driven decision-making and automation, making AI applications more adaptive and autonomous.

**Setting Up API Keys** for Hugging Face, Google, and OpenAI
Before running the code examples in this chapter, API keys must be configured for Hugging Face Hub, Google APIs (Serper and Gemini), and OpenAI. This script retrieves stored credentials in Google Colab Secrets and sets them as environment variables for seamless integration with AI services.

***Note:*** See Google Colab Secrets for instructions on how to store and manage API keys securely.

In [ ]:
# API Key Setup for Hugging Face Hub, Google API, and OpenAI in Google Colab
# Constants and API Key Configuration
import os
from google.colab import userdata

# === Load API keys securely from Google Colab Secrets ===
def load_api_keys():
    keys = {
        "HF_TOKEN": userdata.get("HF_TOKEN"),
        "OPENAI_API_KEY": userdata.get("OPENAI_API_KEY"),
        "SERPAPI_API_KEY": userdata.get("SERPAPI_API_KEY"),
        "SERPER_API_KEY": userdata.get("SERPER_API_KEY")
    }
    for key, value in keys.items():
        if not value:
            raise ValueError(f"❌ Missing {key}. Please set this API key in Colab secrets.")
        os.environ[key] = value
    print("✅ All API keys loaded and configured successfully.")

# Execute API key loading upon running this cell
load_api_keys()


### Listing 7-1: Loading and Analyzing Game Data
This code retrieves and processes the **Steam Games** dataset from Hugging Face. It selects the top five highest-rated games, sorting them by positive reviews. The dataset provides structured information on game titles, genres, and player feedback, forming the foundation for AI-driven analysis and decision-making.

***Note:*** Before running this code, ensure you have the necessary dependencies installed by running:

In [ ]:
%pip install --quiet datasets

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the Steam Games dataset from Hugging Face
dataset = load_dataset("FronkonGames/steam-games-dataset",
                       split="train")

# Convert to Pandas DataFrame
df = dataset.to_pandas()

# Normalize column names (dataset may use lowercase fields)
df.columns = [c.lower() for c in df.columns]

# Select relevant columns and sort by positive reviews
df = df[["name", "genres", "positive"]].sort_values(
     by="positive", ascending=False).head(5)

# Rename columns for clarity
df.columns = ["Game", "Genres", "Positive_Reviews"]

# Display structured dataset output
print("\nTop 5 Highest Rated Games on Steam:")
print(df.to_string(index=False))

### Listing 7-2: Building Prompt Templates and Querying AI Models with LangChain

This listing walks through a progression of prompt-engineering techniques using
LangChain and Hugging Face models. It begins by installing the required packages
and setting up a Hugging Face text-generation endpoint. The first cells introduce
basic prompt templates, then extend them with one-shot and few-shot examples.
Next, the code demonstrates variable substitution to create flexible, category-
driven trivia prompts. The listing then integrates structured data from the Steam
Games Dataset, showing how real-world information can be fed into AI-driven game
concept generation. Finally, the listing runs all templates end-to-end, producing
structured responses from several prompt formats.

***Note:*** Install the required dependencies before running the listing. The pip installation step may produce a few warnings or version-conflict messages in Colab, but in our experience the code cells still run correctly once the packages are in place.

In [ ]:
# Install required packages for Hugging Face and LangChain usage

print("Installing packages... this can take a minute or two.")

%pip install -q langchain langchain-community langchain-huggingface langchain-openai google-search-results

print("All required packaged installed and ready!")

In [ ]:
# === LangChain chat model using Hugging Face's router ===

import os

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Pick a provider-backed open model exposed through Hugging Face
DEFAULT_MODEL = "openai/gpt-oss-20b"
TEMP = 0.2  # 0.0 = focused/deterministic, 1.0 = more creative/variable

# The router forwards requests to a hosted model provider
HUGGING_FACE_ROUTER_URL = "https://router.huggingface.co/v1"

# Use LangChain with Hugging Face's OpenAI-compatible router
chat_llm = ChatOpenAI(
    model=DEFAULT_MODEL,              # Hugging Face model id
    base_url=HUGGING_FACE_ROUTER_URL,
    api_key=os.environ["HF_TOKEN"],   # Loaded from Colab secrets
    temperature=TEMP,                 # Lower = more focused answers
    max_tokens=1024,                   # Max tokens in each reply
)

parser = StrOutputParser()            # Parse output into a string


# === Helper to run any prompt template ===

def ask(chain, question=None, **kwargs):
    """Invoke a chain with an optional question and other variables."""
    inputs = {}
    if question is not None:
        inputs["question"] = question
    inputs.update(kwargs)
    return chain.invoke(inputs)


# === Quick sanity check ===

raw_response = chat_llm.invoke(
    "Answer in one sentence: What is the tallest mountain on Earth?"
)

print("Raw endpoint response:")
print(repr(raw_response))

In [ ]:
# === Basic Trivia Prompt Example ===

basic_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a trivia expert. Answer clearly and succintly.",
    ),
    (
        "human",
        "{question}",
    ),
])

# Build the chain (prompt -> LLM -> text)
basic_chain = basic_prompt | chat_llm | parser

# Invoke the chain with a trivia question
response = basic_chain.invoke({
    "question": "What causes a solar eclipse?",
})

print("📘 Trivia Response:\n", response)

In [ ]:
# === Few-shot trivia example with Hugging Face ===

few_shot_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a trivia expert. Answer clearly and concisely.",
    ),
    (  # Example 1
        "human", "What is gravity?",
    ),
    (
        "ai",
        "Answer: Gravity pulls objects together. "
        "Why it matters: It explains motion on Earth and in space.",
    ),
    (  # Example 2
        "human", "What is DNA?",
    ),
    (
        "ai",
        "Answer: DNA carries genetic instructions. "
        "Why it matters: It shapes how organisms grow and function.",
    ),
    # (Optional) Add more examples here to reinforce the pattern
    (
        "human", "{question}",
    ),
])

few_shot_chain = few_shot_prompt | chat_llm | parser

response = ask(
    few_shot_chain,
    "What causes a solar eclipse?",
)

print("📗 Few-shot Response:\n", response)

In [ ]:
# === Variable trivia prompt with Hugging Face ===

variable_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a trivia expert. Answer clearly and concisely.",
    ),
    (
        "human",
        "Category: {trivia_category}",
    ),
    (
        "human",
        "Known fact: {known_fact}",
    ),
    (
        "human",
        "Question: {question}. Use the known fact in your "
        "answer. Keep it under 50 words.",
    ),
])

# Build the chain (prompt -> LLM -> text)
variable_chain = variable_prompt | chat_llm | parser

# Without additional context
response_no_fact = ask(
    variable_chain,
    "Why can't we see stars during the day?",
    trivia_category="Astronomy",
    known_fact="None provided.",
)

# With contextual fact
response_with_fact = ask(
    variable_chain,
    "Why can't we see stars during the day?",
    trivia_category="Astronomy",
    known_fact="It is daytime in Raleigh.",
)

print("🔹 Without Fact:\n", response_no_fact)
print("\n🔸 With Fact:\n", response_with_fact)

In [ ]:
# === Game analysis with Hugging Face ===

# Format the games into a compact list for the prompt
game_list = "\n".join(
    f"{row['Game']} (Genre: {row['Genres']}, "
    f"Reviews: {row['Positive_Reviews']})"
    for _, row in df.iterrows()
)

# Create the game-analysis prompt
game_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Using the top games below, design a simple three-agent game.",
    ),
    (
        "human",
        "Top-rated games:\n{game_list}\n\n"
        "Provide:\n"
        "- A recommended genre\n"
        "- A game title\n"
        "- A basic mechanic\n"
        "- Three agent roles (2 players, 1 GM)\n"
        "- A short play explanation",
    ),
])

# Build the chain (prompt -> LLM -> text)
game_chain = game_prompt | chat_llm | parser

# Invoke using the shared helper
response = ask(
    game_chain,
    game_list=game_list,
)

print("🎮 Game Concept:\n", response)

### Listing 7-3: Implementing AI Agents and Automating a Trivia Contest
This code sets up Neural Duel, a structured AI-driven trivia game using CrewAI. It defines four agents—a Game Master, two contestants, and a Judge—each assigned specific tasks. The Game Master generates real-time trivia questions with a web search tool, the contestants respond independently, and the Judge evaluates and declares the winner. The flow function ensures data moves correctly between agents, orchestrating a fair and competitive game.

**Note:** Before running this code, install the required dependencies  
(for example, `pip install "crewai[tools]" langchain-openai openai`).  
Because open-source packages evolve quickly, Colab may show occasional
version warnings or dependency conflicts during installation. These seldom
prevent the code from running, but if something does break, try restarting
the runtime, reinstalling packages, or switching to an alternate search tool
from `crewai_tools`. If the agents appear to loop or stall, increasing
`max_iter` to 3 and enabling `verbose=True` usually helps stabilize behavior
and makes debugging easier.

In [ ]:
%%capture --no-stderr
%pip install crewai crewai-tools langchain-openai openai qdrant-client

print("----- CrewAI Installed! -----")

In [ ]:
import re
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI
from crewai_tools import SerperDevTool

# --- Initialize Web Search Tool ---
search_tool = SerperDevTool()

# --- Define LLMs ---
system_llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7,
                        max_tokens=250)
player1_llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.8,
                         max_tokens=50)
player2_llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.9,
                         max_tokens=50)

# --- Define Agents ---

game_master = Agent(
    role="Game Master",
    goal="Generate trivia questions on non-political current events today.",
    backstory="An impartial host creating fair and engaging trivia games.",
    tools=[search_tool],
    max_iter = 2,
    llm=system_llm
)

player_1 = Agent(
    role="Contestant 1",
    goal="Answer trivia questions quickly using only internal knowledge.",
    backstory="A fast-thinking trivia player relying on existing knowledge.",
    max_iter = 2,
    llm=player1_llm
)

player_2 = Agent(
    role="Contestant 2",
    goal="Answer trivia questions strategically, focusing on reasoning.",
    backstory="A methodical contestant carefully formulating responses.",
    max_iter = 2,
    llm=player2_llm
)

judge = Agent(
    role="Judge",
    goal="Evaluate trivia answers and declare a winner with justification.",
    backstory="An impartial adjudicator ensuring fairness in competition.",
    max_iter = 2,
    llm=system_llm
)

# --- Define Tasks ---

generate_question_and_answer = Task(
    description="Create a trivia question on a non-political event today. "
                "Ensure the event is recent and unlikely in LLM training data. "
                "Format the response as:\n\n"
                "Question: <trivia question>\n"
                "Answer: <correct answer>",
    expected_output="A clearly formatted trivia question and correct answer.",
    agent=game_master
)

player1_answer = Task(
    description="Given the question:\n{question}\n\n"
                "Provide your best answer using only internal knowledge.",
    expected_output="A concise and accurate response to the trivia question.",
    agent=player_1
)

player2_answer = Task(
    description="Given the question:\n{question}\n\n"
                "Provide your best answer with logical reasoning.",
    expected_output="A thoughtful and well-reasoned response to the trivia "
                    "question.",
    agent=player_2
)

evaluate_and_declare_winner = Task(
    description="Given the question:\n{question}\n\n"
                "Correct answer:\n{right_answer}\n\n"
                "Player 1's answer:\n{player1_answer}\n\n"
                "Player 2's answer:\n{player2_answer}\n\n"
                "Determine the most accurate response and declare the winner.\n\n"
                "Format as:\n\n"
                "Winner: Contestant X\n"
                "Justification: <brief explanation>",
    expected_output="A clear winner declaration and reasoning.",
    agent=judge
)

# --- Define Crews ---

game_master_crew = Crew(
    agents=[game_master],
    tasks=[generate_question_and_answer],
    verbose=True
)

player1_crew = Crew(
    agents=[player_1],
    tasks=[player1_answer],
    verbose=True
)

player2_crew = Crew(
    agents=[player_2],
    tasks=[player2_answer],
    verbose=True
)

judge_crew = Crew(
    agents=[judge],
    tasks=[evaluate_and_declare_winner],
    verbose=True
)

# --- Define Flow Function to Orchestrate Contest ---

def extract_question_answer(response_text):
    """Extracts question and answer from structured text using regex."""
    match = re.search(r"Question:\s*(.*?)\s*Answer:\s*(.*)",
                      response_text, re.IGNORECASE)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    return "No question generated.", "No correct answer provided."

def run_neural_duel():
    """Orchestrates the trivia contest flow between AI agents."""
    print("🎲 Starting Neural Duel!\n")

    # Step 1: Game Master generates trivia question and correct answer
    gm_results = game_master_crew.kickoff()
    print("\n🔍 Raw Game Master Results:\n", gm_results)

    # Extract question and correct answer
    question, correct_answer = extract_question_answer(str(gm_results))

    print(f"\n📝 Trivia Question: {question}")
    print(f"🤫 (Secret) Correct Answer: {correct_answer}")

    # Step 2: Players answer the question
    player1_results = player1_crew.kickoff(inputs={"question": question})
    player2_results = player2_crew.kickoff(inputs={"question": question})

    print(f"\n🏅 Contestant 1's Answer: {str(player1_results)}")
    print(f"🏅 Contestant 2's Answer: {str(player2_results)}")

    # Step 3: Judge evaluates responses and declares the winner
    judge_results = judge_crew.kickoff(inputs={
        "question": question,
        "right_answer": correct_answer,
        "player1_answer": str(player1_results),
        "player2_answer": str(player2_results)
    })

    print("\n🏆 Neural Duel Results:\n", str(judge_results))

# --- Run the Game ---
run_neural_duel()



### Listing 7-4: Generating Chapter Artwork with the Illustrator Agent
This code defines the Chapter Illustrator agent, a lightweight AI designed to create a single image for the chapter. The agent uses DALL·E as its sole tool and receives a short visual prompt describing the desired scene. When run, the agent produces a landscape-oriented, comic-style illustration of two AI robots playing chess, demonstrating how an agent can carry out a focused creative task within a larger workflow.

In [ ]:
%%capture --no-stderr
%pip install crewai crewai-tools langchain-openai openai qdrant-client

In [ ]:
import os
from crewai import Agent, Task, Crew, Process
from crewai_tools import DallETool

# Ensure API key is set
if "OPENAI_API_KEY" not in os.environ:
    raise SystemExit("Please set the OPENAI_API_KEY env var before running.")

# Landscape DALL·E tool
image_tool = DallETool(
    model="dall-e-3",
    size="1792x1024",   # landscape orientation
    quality="standard",
    n=1,
)

# Minimal and clear agent
illustrator_agent = Agent(
    role="Illustrator",
    goal="Generate exactly one DALL-E image when requested.",
    backstory="You convert short prompts into visual concepts.",
    tools=[image_tool],
    llm="gpt-4o-mini",
    verbose=True,
)

# Task: explicitly tell agent how to call the tool
chapter_image_task = Task(
    description=(
        "Use ONLY the Dall-E Tool.\n\n"
        "Your Action Input MUST be JSON of this form:\n"
        "{\"image_description\": \"<prompt>\"}\n\n"
        "Use this image description:\n"
        "Create a clean, comic-style image on a white background with a "
        "color palette of Primary Red, Bright Blue, Sunny Yellow, Deep Black, "
        "White, and Muted Grey. Depict two playful AI bots playing chess against "
        "each other. Landscape composition."
    ),
    expected_output="A DALL-E image URL.",
    agent=illustrator_agent,
)

chapter_image_crew = Crew(
    agents=[illustrator_agent],
    tasks=[chapter_image_task],
    process=Process.sequential,
    verbose=True,
)

def generate_chapter_image():
    result = chapter_image_crew.kickoff()
    print("\n=== Chapter Image Result ===\n")
    print(result)
    print("\n============================\n")

if __name__ == "__main__":
    generate_chapter_image()

### Listing 7-20 – AI-Assisted Mermaid Diagram Generator

This code demonstrates how to generate structured Mermaid diagrams using an AI-assisted workflow. It defines editable chart settings such as diagram direction, hierarchy, and visual style at the top for easy reuse across different diagrams. The program uses an OpenAI model through LangChain to convert a plain-text hierarchy into a structured diagram specification, then renders the specification deterministically into Mermaid syntax with consistent node styling and labeling. The process includes defining reusable visual themes, assigning hierarchical node identifiers automatically, and saving the resulting Mermaid file for documentation or publication use.
**NOTE:** Be sure your OpenAI API key is stored in Google Colab Secrets as `OPENAI_API_KEY`. Also run the `pip install` cell below to install the required libraries before executing this listing.

In [ ]:
%pip install -q langchain-openai langchain-core mermaid-py pydantic

In [ ]:
import string
from typing import List

from google.colab import userdata
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# =========================================================
# CHART-SPECIFIC SETTINGS: edit these for each new chart
# =========================================================
CHART_NAME = "model_tuning_pipeline"
CHART_DIRECTION = "TB"   # LR or TB

CHART_HIERARCHY = """
Agentic AI
  - Defining Agents
    - Games
    - Automation
    - Abstractions
  - Langchain
    - Prompts
    - Models
    - Role
    - Mission
  - CrewAI
    - Tools
    - Tasks
    - Flows
    - Collaboration
  - Agent Applications
    - Simulations
    - Assistants
    - Autonomous Systems
"""

CHART_STYLE_INTENT = """
Use a root box, major step boxes, and smaller supporting boxes.
Keep labels concise and suitable for Mermaid.
"""

OUTPUT_FILE = f"{CHART_NAME}.mmd"

# =========================================================
# MODEL / APP SETTINGS: usually stable
# =========================================================
MODEL = "gpt-5-mini"
TEMP = 0.2
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

# =========================================================
# VISUAL THEME: reuse across many charts
# =========================================================
FONT_SIZE_ROOT = "32px"
FONT_SIZE_MAJOR = "28px"
FONT_SIZE_MINOR = "24px"

ROOT_FILL = "#ff0000"
MAJOR_FILL = "#0000f0"
MINOR_FILL = "#ffff00"

ROOT_TEXT = "#ffffff"
MAJOR_TEXT = "#ffffff"
MINOR_TEXT = "#000000"

STROKE = "#000000"
STROKE_WIDTH = "2px"

# =========================================================
# STRUCTURED SCHEMA
# =========================================================
class DiagramNode(BaseModel):
    label: str = Field(..., description="Text shown in the Mermaid node")
    children: List["DiagramNode"] = Field(default_factory=list)

DiagramNode.model_rebuild()

class DiagramSpec(BaseModel):
    direction: str = Field(..., description="Either LR or TB")
    root: DiagramNode

# =========================================================
# LLM
# =========================================================
llm = ChatOpenAI(
    model=MODEL,
    temperature=TEMP,
    api_key=OPENAI_API_KEY,
)

structured_llm = llm.with_structured_output(DiagramSpec)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You generate a simple hierarchical diagram spec for Mermaid.

Return only valid structured data.

Rules:
- direction must be LR or TB
- produce a single root node
- preserve the user's hierarchy faithfully
- keep labels concise
- you may include <br> where useful for line breaks
- do not invent extra branches unless clearly implied"""
    ),
    (
        "user",
        """Create a diagram spec from this request.

Direction: {direction}

Hierarchy:
{hierarchy}

Style intent:
{style_intent}
"""
    ),
])

chain = prompt | structured_llm

# =========================================================
# MERMAID RENDERER
# =========================================================
def style_line(node_id: str, depth: int) -> str:
    if depth == 0:
        fill, color, size = ROOT_FILL, ROOT_TEXT, FONT_SIZE_ROOT
    elif depth == 1:
        fill, color, size = MAJOR_FILL, MAJOR_TEXT, FONT_SIZE_MAJOR
    else:
        fill, color, size = MINOR_FILL, MINOR_TEXT, FONT_SIZE_MINOR

    return (
        f"style {node_id} "
        f"fill:{fill},stroke:{STROKE},stroke-width:{STROKE_WIDTH},"
        f"color:{color},font-size:{size}"
    )

def assign_ids(root: DiagramNode):
    """
    Root gets A.
    Root children get B, C, D...
    Descendants get parent-based suffixes like B1, B2, C1...

    Edge style:
    - depth 0 node branches to each major child
    - depth 1+ children form a chained mini workflow
    """
    nodes = []
    edges = []

    def walk(node: DiagramNode, node_id: str, depth: int):
        nodes.append((node_id, node.label, depth))

        child_ids = []

        if depth == 0:
            for i, child in enumerate(node.children):
                child_id = string.ascii_uppercase[i + 1]  # B, C, D...
                child_ids.append((child, child_id))
                edges.append((node_id, child_id))  # A --> B, A --> C, ...
        else:
            for i, child in enumerate(node.children, start=1):
                child_id = f"{node_id}{i}"
                child_ids.append((child, child_id))

            if child_ids:
                edges.append((node_id, child_ids[0][1]))  # B --> B1
                for i in range(len(child_ids) - 1):
                    edges.append((child_ids[i][1], child_ids[i + 1][1]))  # B1 --> B2

        for child, child_id in child_ids:
            walk(child, child_id, depth + 1)

    walk(root, "A", 0)
    return nodes, edges

def render_mermaid(spec: DiagramSpec) -> str:
    nodes, edges = assign_ids(spec.root)
    label_map = {node_id: label for node_id, label, _ in nodes}

    lines = [f"graph {spec.direction}"]

    for node_id, _, depth in nodes:
        lines.append(style_line(node_id, depth))

    lines.append("")

    emitted_nodes = set()
    for src, dst in edges:
        src_repr = f"{src}[{label_map[src]}]" if src not in emitted_nodes else src
        dst_repr = f"{dst}[{label_map[dst]}]" if dst not in emitted_nodes else dst
        lines.append(f"{src_repr} --> {dst_repr}")
        emitted_nodes.add(src)
        emitted_nodes.add(dst)

    return "\n".join(lines)

# =========================================================
# OPTIONAL RENDERING
# =========================================================
RENDER_NOTEBOOK = True
SAVE_PNG = True
PNG_FILE = OUTPUT_FILE.replace(".mmd", ".png")

def display_mermaid_notebook(mermaid_text: str):
    import mermaid as md
    from IPython.display import display

    diagram = md.Mermaid(mermaid_text)
    display(diagram)

def save_mermaid_png(mermaid_text: str, png_file: str):
    import os
    import base64
    import requests

    # mermaid-py documents mermaid.ink as the default backend server
    server = os.environ.get("MERMAID_INK_SERVER", "https://mermaid.ink")
    graph_bytes = mermaid_text.encode("utf-8")
    encoded = base64.urlsafe_b64encode(graph_bytes).decode("ascii")
    url = f"{server}/img/{encoded}"

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with open(png_file, "wb") as f:
        f.write(response.content)

    print(f"Saved {png_file}")

# =========================================================
# RUN
# =========================================================
spec = chain.invoke({
    "direction": CHART_DIRECTION,
    "hierarchy": CHART_HIERARCHY,
    "style_intent": CHART_STYLE_INTENT,
})

mermaid_text = render_mermaid(spec)

print("=== STRUCTURED SPEC ===")
print(spec.model_dump_json(indent=2))

print("\n=== MERMAID ===")
print(mermaid_text)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(mermaid_text)

print(f"\nSaved {OUTPUT_FILE}")

if RENDER_NOTEBOOK:
    try:
        display_mermaid_notebook(mermaid_text)
    except Exception as e:
        print(f"Notebook render skipped: {e}")

if SAVE_PNG:
    try:
        save_mermaid_png(mermaid_text, PNG_FILE)
    except Exception as e:
        print(f"PNG render skipped: {e}")